In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.model_selection import train_test_split

from src.features import FEATURE_NAMES, normalize_features
from src.models import train_model, predict, get_feature_importance
from src.evaluate import compute_correlations

sns.set_theme(style='whitegrid')
np.random.seed(42)

In [ ]:
X = np.load('../outputs/predictions/X_features.npy')
y = np.load('../outputs/predictions/y_mrr10.npy')

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {X_train.shape}  Test: {X_test.shape}')
print(f'Features: {FEATURE_NAMES}')

In [ ]:
# Index:  0=max_sim  1=sim_variance  2=high_sim_count  3=rank_dropoff  4=emb_variance
#         5=term_overlap  6=query_length  7=query_idf_sum  8=query_entropy
#         9=clarity  10=wig  11=nqc

GROUPS = {
    'All Features':          list(range(12)),
    '- Similarity':          list(range(5, 12)),
    '- Embedding':           [0, 1, 2, 3] + list(range(5, 12)),
    '- Lexical':             list(range(0, 5)) + list(range(9, 12)),
    '- Traditional QPP':     list(range(0, 9)),
}

records = []
for label, cols in GROUPS.items():
    Xtr_n, Xte_n, _ = normalize_features(X_train[:, cols], X_test[:, cols])
    model  = train_model('random_forest', Xtr_n, y_train, cv=False)
    y_pred = predict(model, Xte_n)
    corr   = compute_correlations(y_pred, y_test)
    records.append({
        'Feature Setting': label,
        'Spearman':  round(corr['spearman_rho'], 4),
        'Pearson':   round(corr['pearson_r'],    4),
        'Kendall':   round(corr['kendall_tau'],  4),
    })
    print(f'{label:25s}  Pearson={corr["pearson_r"]:.4f}  Spearman={corr["spearman_rho"]:.4f}')

df_ablation = pd.DataFrame(records).set_index('Feature Setting')
df_ablation.to_csv('../outputs/predictions/ablation_results.csv')

In [ ]:
settings = list(df_ablation.index)
x = np.arange(len(settings))

plt.figure(figsize=(10, 4))
plt.plot(x, df_ablation['Pearson'],  'o-', color='#1E88E5', lw=2, label='Pearson r')
plt.plot(x, df_ablation['Spearman'], 's-', color='#43A047', lw=2, label='Spearman rho')
plt.plot(x, df_ablation['Kendall'],  '^-', color='#FB8C00', lw=2, label='Kendall tau')
plt.xticks(x, settings, rotation=15, ha='right')
plt.ylabel('Correlation')
plt.title('Feature Ablation — QPP Prediction Accuracy')
plt.legend()
plt.tight_layout()
plt.savefig('../outputs/figures/ablation_correlations.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
X_train_n, X_test_n, _ = normalize_features(X_train, X_test)
model_full = train_model('random_forest', X_train_n, y_train, cv=False)
imp        = get_feature_importance(model_full)
imp_df     = pd.DataFrame(sorted(imp.items(), key=lambda x: x[1]),
                          columns=['Feature', 'Importance'])

def feature_color(name):
    if name in ('emb_variance', 'high_sim_count', 'sim_variance', 'rank_dropoff', 'max_sim'):
        return '#E53935'
    if name in ('term_overlap', 'query_length', 'query_idf_sum', 'query_entropy'):
        return '#78909C'
    return '#FB8C00'

plt.figure(figsize=(8, 5))
plt.barh(imp_df['Feature'], imp_df['Importance'],
         color=[feature_color(f) for f in imp_df['Feature']], edgecolor='white')
for i, (feat, val) in enumerate(zip(imp_df['Feature'], imp_df['Importance'])):
    plt.text(val + 0.001, i, f'{val:.3f}', va='center', fontsize=8)
plt.xlabel('Relative Feature Importance')
plt.title('Per-Feature Importance — Random Forest')
patches = [
    mpatches.Patch(color='#E53935', label='Semantic / Similarity'),
    mpatches.Patch(color='#FB8C00', label='Traditional QPP'),
    mpatches.Patch(color='#78909C', label='Lexical'),
]
plt.legend(handles=patches)
plt.tight_layout()
plt.savefig('../outputs/figures/feature_importance_ablation.png', dpi=150, bbox_inches='tight')
plt.show()